<a href="https://colab.research.google.com/github/SMK-BTW/Twenty/blob/main/Machine_Learning_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q --upgrade --force-reinstall pandas pyarrow datasets huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.6/111.6 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 65.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.0/120.0 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.4/110.4 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.9/203.9 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
!pip install -q duckdb

In [ ]:
from huggingface_hub import hf_hub_url
import duckdb

In [ ]:
url = hf_hub_url(
    repo_id="yandex/yambda",
    filename="flat/50m/multi_event.parquet",
    repo_type="dataset"
)


https://huggingface.co/datasets/yandex/yambda/resolve/main/flat/50m/multi_event.parquet


In [ ]:
con = duckdb.connect()

con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")

In [ ]:
df = con.execute(f"""
    SELECT *
    FROM read_parquet('{url}')
    USING SAMPLE 100000 ROWS
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
print(df.shape)
print("Unique users:", df["uid"].nunique())
print("Unique tracks:", df["item_id"].nunique())

print("\nEvent types:")
print(df["event_type"].value_counts())

(100000, 7)
Unique users: 8105
Unique tracks: 47124

Event types:
event_type
listen       97368
like          1773
unlike         605
dislike        218
undislike       36
Name: count, dtype: int64


In [ ]:
df.head()

,uid,timestamp,item_id,is_organic,played_ratio_pct,track_length_seconds,event_type
0,634000,8590595,7017693,0,0,190,listen
1,898400,14131650,3827963,0,21,135,listen
2,684700,22771895,7948836,0,100,250,listen
3,154200,24567735,2936626,0,100,215,listen
4,1600,15968260,9076860,1,100,165,listen


played_ratio_pct is the percentage of track played (1-100)

In [ ]:
listening_df = df[
    (df["event_type"] == "listen") &
    (df["played_ratio_pct"] > 50)
].copy()

In [ ]:
print("Meaningful listening events:", len(listening_df))
print("Unique users:", listening_df["uid"].nunique())
print("Unique songs:", listening_df["item_id"].nunique())

Meaningful listening events: 63181
Unique users: 7191
Unique songs: 33887


We have meaningful listen data but the user might have listened to one track several times so we collapse them

In [ ]:
interactions = (
    listening_df[
        ["uid", "item_id"]
    ]
    .drop_duplicates()
    .copy()
)

interactions["interaction"] = 1

interactions.head()

,uid,item_id,interaction
2,684700,7948836,1
3,154200,2936626,1
4,1600,9076860,1
5,575200,2675456,1
6,308400,178446,1


In [ ]:
print("User-song pairs:", len(interactions))
print("Users:", interactions["uid"].nunique())
print("Songs:", interactions["item_id"].nunique())

User-song pairs: 60868
Users: 7191
Songs: 33887


In [ ]:
user_activity = (
    interactions.groupby("uid")
    .size()
    .sort_values(ascending=False)
)

user_activity.head(10)

uid
7200    1079
1400     859
800      638
1600     510
1900     488
700      477
7100     402
2300     374
1100     356
600      347
dtype: int64

In [ ]:
target_user = user_activity.index[0]

print("Sample target user:", target_user)
print(
    "Number of unique songs:",
    user_activity.loc[target_user]
)

Sample target user: 7200
Number of unique songs: 1079


Sample input Data for the proposal

In [ ]:
sample_input = (
    listening_df[
        listening_df["uid"] == target_user
    ][
        [
            "uid",
            "timestamp",
            "item_id",
            "played_ratio_pct",
            "event_type"
        ]
    ]
    .head(10)
)

sample_input

,uid,timestamp,item_id,played_ratio_pct,event_type
49,7200,2636820,7461360,100,listen
122,7200,11300115,3785541,100,listen
310,7200,14236415,8750380,100,listen
453,7200,2699490,3411963,100,listen
500,7200,4525330,5102121,100,listen
591,7200,6443200,2369699,100,listen
634,7200,17472555,5155988,100,listen
642,7200,14673015,8273567,100,listen
739,7200,14185025,8435723,100,listen
779,7200,16005850,5358396,100,listen


K-NN Model Input

In [ ]:
sample_model_input = interactions[
    interactions["uid"] == target_user
].copy()

sample_model_input.head(10)

,uid,item_id,interaction
49,7200,7461360,1
122,7200,3785541,1
310,7200,8750380,1
453,7200,3411963,1
500,7200,5102121,1
591,7200,2369699,1
634,7200,5155988,1
642,7200,8273567,1
739,7200,8435723,1
779,7200,5358396,1


In [ ]:
OUTPUT_COLUMNS = [
    "uid",
    "rank",
    "item_id",
    "score"
]

print("Expected recommender output columns:")
print(OUTPUT_COLUMNS)

Expected recommender output columns:
['uid', 'rank', 'item_id', 'score']


In [ ]:
already_listened = set(
    interactions.loc[
        interactions["uid"] == target_user,
        "item_id"
    ]
)

print("Target user:", target_user)
print("Songs already listened to:", len(already_listened))

Target user: 7200
Songs already listened to: 1079


In [ ]:
def validate_recommendations(
    recommendations,
    already_listened
):
    overlap = set(
        recommendations["item_id"]
    ).intersection(already_listened)


    if len(overlap) > 0:
        return f"Invalid - overlapping songs: {overlap}"
    else:
        return "Valid"